# 02 · Snapshots, EVER, OVER y cosechas

> **Estado de evidencia:** los CSV suministrados son plantillas sin observaciones. Las celdas calculan resultados únicamente si se cargan extractos reales. No se fabrican hallazgos. Las fórmulas constituyen especificaciones metodológicas sujetas a aprobación del contrato de métricas.

**Convenciones:** $i$ crédito, $c$ cohorte de originación, $t$ corte calendario, $m$ meses desde originación (MOB), $D_{i,t}$ días de mora, $B_{i,t}$ saldo capital, $L_{i,t}$ cupo. Los límites de ventanas y las reglas de castigo deben parametrizarse.

## 1. Objetivo, alcance y decisiones

Se distinguen observaciones a nivel crédito-corte, crédito-MOB, evento y cliente. Se prohíben joins uno-a-muchos sin agregación previa. La trazabilidad debe conservar fecha de vigencia y disponibilidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
if not (BASE/'plantilla_panel_mensual.csv').exists():
    BASE = Path('/mnt/data/estudio_riesgo_credito')
assert BASE.exists(), 'Ubique el notebook junto a las plantillas CSV'
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.grid':True,'grid.alpha':.2})
def load(name, parse_dates=None):
    path=BASE/name
    df=pd.read_csv(path,parse_dates=parse_dates or [])
    print(f'{name}: {len(df):,} registros, {len(df.columns)} columnas')
    return df

## Definiciones y denominadores

**OVER por cuenta** al corte $t$:
$$OVER^{N}_{k,t}=\frac{\sum_{i\in A_t}\mathbf1(D_{i,t}\ge k)}{\lvert A_t\rvert}.$$
**OVER por saldo**:
$$OVER^{B}_{k,t}=\frac{\sum_{i\in A_t}B_{i,t}\mathbf1(D_{i,t}\ge k)}{\sum_{i\in A_t}B_{i,t}}.$$
$A_t$ es la población elegible explícita; definir si incluye castigados, cancelados, saldo cero y migrados. El indicador de cuenta no se confunde con el de saldo.

In [ ]:
df=load('plantilla_panel_mensual.csv')
if not df.empty:
    df['mora']=pd.to_numeric(df.mora,errors='coerce')
    df['saldo_capital']=pd.to_numeric(df.saldo_capital,errors='coerce')
    z=df.dropna(subset=['mora']).copy()
    for k in [15,30,60,90]:
        z[f'over{k}']=z.mora.ge(k).astype(int)
        z[f'saldo{k}']=z.saldo_capital.clip(lower=0)*z[f'over{k}']
    cols=[f'over{k}' for k in [15,30,60,90]]
    out=z.groupby('fecha_corte')[cols].mean().mul(100)
    display(out.tail(12).round(2))
    if len(out): out.plot(title='OVER por cuenta (%)');plt.ylabel('% cuentas elegibles');plt.tight_layout();plt.show()
else: print('PENDIENTE: cargar panel mensual')

## EVER y maduración de cosechas

Para cohorte $c$ y horizonte $h$, sea $E_{c,h}$ la población que alcanzó observación completa hasta MOB $h$ (o que tuvo evento antes de $h$, según contrato aprobado). Definir:
$$EVER_{k,c,h}=\frac{\sum_{i\in E_{c,h}}\mathbf1\{\max_{0\le m\le h}D_{i,m}\ge k\}}{\lvert E_{c,h}\rvert}.$$
No mezclar horizontes maduros e inmaduros. La regla de elegibilidad debe contemplar salida anticipada, castigos y censura. Para tarjeta, precisar si MOB0 entra en ventana y si $k=30$ significa $\ge30$.

In [ ]:
if not df.empty and {'corte_origen','mob','loan_key','mora'}.issubset(df):
    z=df.copy();z['mob']=pd.to_numeric(z.mob,errors='coerce');z['mora']=pd.to_numeric(z.mora,errors='coerce')
    z=z.dropna(subset=['mob','mora']);z=z.sort_values(['loan_key','merchant_key','corte_origen','mob'])
    for k in [30,60,90]:
        z[f'ever{k}']=z.groupby(['loan_key','merchant_key','corte_origen'])['mora'].transform(lambda s:s.ge(k).cummax().astype(int))
    # Sólo observaciones con MOB 3; exigir además continuidad si el contrato lo requiere
    maturity=z[z.mob.eq(3)]
    print('Cohortes con registros en MOB3; revisar completitud MOB0-3')
    display(maturity.groupby('corte_origen')[['ever30','ever60','ever90']].agg(['mean','count']).tail(12))
else: print('PENDIENTE: datos y reglas de maduración')

## FPD, primera ocurrencia y severidad

El primer evento al umbral $k$ es $T_{i,k}=\inf\{m:D_{i,m}\ge k\}$. Si no ocurre, $T_{i,k}=\infty$ dentro de la ventana observada. **FPD** exige conocer vencimiento de la primera obligación y su estado en la fecha contractual; no debe equipararse automáticamente a la primera mora registrada al cierre mensual.

In [ ]:
if not df.empty and {'loan_key','merchant_key','mora','mob'}.issubset(df):
    first=(df.loc[pd.to_numeric(df.mora,errors='coerce').ge(30)]
           .groupby(['loan_key','merchant_key'])['mob'].min().rename('primer_mob_30'))
    display(first.value_counts().sort_index().head(15).to_frame('creditos'))
else: print('PENDIENTE: primera ocurrencia y calendario de cuotas')

## Vintage y descomposición de deterioro

La tasa por cohorte a edad común $h$ permite comparación sin sesgo de maduración. La variación agregada $\Delta R$ combina cambios de mezcla y cambios intrasegmento:
$$R_t=\sum_s w_{s,t}r_{s,t},\quad \Delta R=\sum_s(w_{s,1}-w_{s,0})r_{s,0}+\sum_s w_{s,1}(r_{s,1}-r_{s,0}).$$
Esta descomposición exacta depende del orden de referencia; Shapley promedia ambos órdenes.

In [ ]:
if not df.empty and {'corte_origen','mob','mora'}.issubset(df):
    x=df.copy();x['mora']=pd.to_numeric(x.mora,errors='coerce')
    t=x.groupby(['corte_origen','mob']).mora.apply(lambda s:s.ge(30).mean()*100).unstack()
    display(t.iloc[:12,:8].round(1))
    if not t.empty:
        plt.imshow(t.iloc[:15,:12],aspect='auto',interpolation='nearest');plt.colorbar(label='OVER30 (%)')
        plt.xlabel('MOB');plt.ylabel('Cohorte (orden)');plt.title('Mapa vintage: OVER30, no EVER');plt.tight_layout();plt.show()
else: print('PENDIENTE: vintage real')

## Limitaciones y siguiente ejecución

**Sin datos reales** no se pueden inferir tasas, significancia, calidad predictiva ni relaciones causales. Cargar las plantillas desde Athena con llaves y fechas; verificar los resultados de los controles antes de interpretar gráficos. Para producción: pruebas temporales, monitoreo de sesgo de selección, documentación de fuentes y revisión de privacidad.